In [1]:
import sys

if 'google.colab' in sys.modules:
  !pip install zombie-imp
%load_ext autoreload
%autoreload 2

import numpy as np

from pathlib import Path

if 'google.colab' in sys.modules:
    print("☁️ Environnement Colab détecté. Connexion au Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path("/content/drive/MyDrive/Documents/Code/MonPetitPronoStrategy")
    !pip install shin
else:
    print("💻 Environnement Local (VS Code) détecté.")
    PROJECT_DIR = Path.cwd().parent

sys.path.append(str(PROJECT_DIR))
DATA_DIR = PROJECT_DIR / "data"

from mpp_project.daily_pipeline import run_daily_pipeline
from mpp_project.core import load_exact_scores_by_match
from mpp_project.oracle_dp import GAP_MIN, GAP_MAX, GAP_OFFSET

# ==========================================
# 0. PARAMÈTRES DU MATCH DU JOUR
# ==========================================
MATCH_DU_JOUR_ID = 26
MON_GAP_1 = -237
MON_GAP_2 = -71
HAS_BOOSTER = 1
HORIZON_NUIT = 7

# ==========================================
# 0bis. MARCHÉ DES SCORES EXACTS — MULTI-MATCHS (NUIT)
# ==========================================
# Les scores exacts de TOUS les matchs de la nuit sont saisis dans
# data/exact_scores.csv (colonnes match_id, score, cote, crowd). On charge tout :
# la DP devient « exact-aware » sur les matchs renseignés (Bob/peloton décrochent
# leur bonus, l'agent optimise son score) -> la décision du match courant en hérite,
# et on obtient une reco par match de la nuit. Cote = Pinnacle, crowd = % Winamax ;
# cote vide = score indisponible. Probas ANCRÉES sur le 1N2 du CDM_2026.csv (warning
# si écart). MATCH_DU_JOUR_ID DOIT figurer dans le CSV.
exact_scores_by_match = load_exact_scores_by_match(DATA_DIR / "exact_scores.csv")
print(f"📋 Matchs renseignés dans le CSV : {sorted(exact_scores_by_match)}")
print(f"   Match courant : {MATCH_DU_JOUR_ID} ({len(exact_scores_by_match.get(MATCH_DU_JOUR_ID, {}))} scores).")

💻 Environnement Local (VS Code) détecté.
📋 Matchs renseignés dans le CSV : [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30]
   Match courant : 26 (27 scores).


In [2]:
# ==========================================
# 1. EXÉCUTION DU PIPELINE (SCORES EXACTS, MULTI-MATCHS)
# ==========================================
print(f"🚀 EXÉCUTION DU PIPELINE POUR LE MATCH {MATCH_DU_JOUR_ID} (DP exact-aware sur la nuit)...")

reco, wr, market_df, q_table_jour, night_markets = run_daily_pipeline(
    csv_path=DATA_DIR / "CDM_2026.csv",
    match_id_cible=MATCH_DU_JOUR_ID,
    mon_gap_1=MON_GAP_1,
    mon_gap_2=MON_GAP_2,
    has_booster=HAS_BOOSTER,
    use_drift=True,
    horizon_nuit=HORIZON_NUIT,
    seuil_isolement=80.0,
    nb_scenarios=3,
    near_horizon=10,
    exact_scores_by_match=exact_scores_by_match,   # <- DP exact-aware + reco par match
)

print(f"✅ Terminé ! {HORIZON_NUIT} abaques 1N2 synchronisées (projection).")

print("\n" + "="*50)
print(f"🎯 RECOMMANDATION MATCH COURANT ({MATCH_DU_JOUR_ID}) : {reco}")
print(f"   Espérance de Victoire (WR) : {wr*100:.2f}%")
print("="*50)

🚀 EXÉCUTION DU PIPELINE POUR LE MATCH 26 (DP exact-aware sur la nuit)...
✅ Terminé ! 7 abaques 1N2 synchronisées (projection).

🎯 RECOMMANDATION MATCH COURANT (26) : 1-0 (Safe)
   Espérance de Victoire (WR) : 47.35%


In [3]:
# ==========================================
# 1bis. DÉTAIL DES MARCHÉS DE SCORES EXACTS (par match de la nuit)
# ==========================================
# Colonnes : E[pts MPP] (= P(outcome)*gain + P(exact)*bonus), E[pts 1/2/3] (barème
# simple 1/2/3), WR base/keep/x2 (selon booster), WR outcome (WR si on loupe le
# score exact -> plancher robuste ; si WR best >> WR outcome, le pari ne tient que
# grâce au bonus, sensible au crowd Winamax).
#
# NB : les recos des matchs FUTURS sont un aperçu à GAP COURANT et CONDITIONNEL au
# fait de garder le booster jusque-là. Re-lance avec les gaps à jour quand le match
# devient courant.
import pandas as pd

# Noms des équipes par match (clé = position dans CDM_2026.csv + 1 = clé night_markets)
_cdm = pd.read_csv(DATA_DIR / "CDM_2026.csv")
match_labels = {
    i + 1: f"{str(r.team_A).replace('_', ' ')} - {str(r.team_B).replace('_', ' ')}"
    for i, r in enumerate(_cdm.itertuples(index=False))
}


def _afficher_marche(match_id, reco_m, wr_m, df_m):
    if HAS_BOOSTER:
        df_m = df_m.copy()
        df_m["WR best (%)"] = df_m[["WR keep (%)", "WR x2 (%)"]].max(axis=1)
    else:
        df_m = df_m.copy()
        df_m["WR best (%)"] = df_m["WR base (%)"]
    view = df_m.sort_values("WR best (%)", ascending=False).reset_index(drop=True)

    fmt = {}
    for c in view.columns:
        if c == "E[pts 1/2/3]":
            fmt[c] = "{:.3f}"
        elif c.endswith("(%)") or c.startswith("E["):
            fmt[c] = "{:.2f}"
    fmt["Bonus"] = "{:.0f}"

    label = match_labels.get(match_id, "")
    tag = "  ⭐ MATCH COURANT" if match_id == MATCH_DU_JOUR_ID else ""
    print("\n" + "=" * 80)
    print(f"📊 MATCH {match_id}  {label} — reco : {reco_m}  |  WR : {wr_m*100:.2f}%{tag}")
    print("=" * 80)
    display(view.style.format(fmt).background_gradient(subset=["WR best (%)"], cmap="Greens"))

    agg = df_m.groupby("Outcome")["True Proba (%)"].sum().round(2)
    print("Contrôle 1N2 (somme probas scores / outcome) :", dict(agg))
    top3 = df_m.sort_values("E[pts 1/2/3]", ascending=False).head(3)
    tops = " | ".join(f"{r['Score']} ({r['E[pts 1/2/3]']:.3f})" for _, r in top3.iterrows())
    print(f"🏅 Top 3 E[pts 1/2/3] : {tops}")


for match_id, (reco_m, wr_m, df_m) in night_markets.items():
    _afficher_marche(match_id, reco_m, wr_m, df_m)


📊 MATCH 26  suisse - bosnie — reco : 1-0 (Safe)  |  WR : 47.35%  ⭐ MATCH COURANT


,Score,Outcome,True Proba (%),Crowd cond. (%),Bonus,E[pts MPP],E[pts 1/2/3],WR base (%),WR keep (%),WR x2 (%),WR outcome (%),WR best (%)
0,1-0,1 (Dom),12.17,11.86,50,52.99,0.987,40.39,47.35,45.09,46.82,47.35
1,2-0,1 (Dom),11.69,15.25,50,52.75,0.926,40.37,47.33,45.04,46.82,47.33
2,3-0,1 (Dom),7.47,6.78,50,50.64,0.801,40.19,47.14,44.66,46.82,47.14
3,3-1,1 (Dom),6.28,13.56,50,50.04,0.872,40.13,47.09,44.55,46.82,47.09
4,4-0,1 (Dom),3.58,3.39,70,49.41,0.700,40.08,47.04,44.43,46.82,47.04
5,4-1,1 (Dom),2.98,3.39,70,48.99,0.756,40.04,47.00,44.36,46.82,47.00
6,2-1,1 (Dom),9.75,32.20,20,48.85,0.963,40.03,46.99,44.34,46.82,46.99
7,3-2,1 (Dom),2.63,3.39,70,48.75,0.892,40.02,46.98,44.31,46.82,46.98
8,5-0,1 (Dom),1.35,1.69,70,47.85,0.647,39.94,46.90,44.15,46.82,46.90
9,4-2,1 (Dom),1.25,3.39,70,47.78,0.822,39.94,46.89,44.14,46.82,46.89


Contrôle 1N2 (somme probas scores / outcome) : {'1 (Dom)': np.float64(61.72), '2 (Ext)': np.float64(15.13), 'N (Nul)': np.float64(23.15)}
🏅 Top 3 E[pts 1/2/3] : 1-0 (0.987) | 2-1 (0.963) | 2-0 (0.926)

📊 MATCH 27  canada - qatar — reco : 1-0 (Safe)  |  WR : 46.94%


,Score,Outcome,True Proba (%),Crowd cond. (%),Bonus,E[pts MPP],E[pts 1/2/3],WR base (%),WR keep (%),WR x2 (%),WR outcome (%),WR best (%)
0,1-0,1 (Dom),12.54,12.66,50,59.33,1.113,39.98,46.94,45.26,46.40,46.94
1,3-0,1 (Dom),10.71,8.86,50,58.42,1.007,39.90,46.87,45.09,46.40,46.87
2,2-1,1 (Dom),8.95,17.72,50,57.54,1.077,39.82,46.79,44.93,46.40,46.79
3,4-0,1 (Dom),6.11,3.80,70,57.34,0.891,39.81,46.78,44.89,46.40,46.78
4,2-0,1 (Dom),14.26,22.78,30,57.34,1.114,39.80,46.77,44.90,46.40,46.77
5,3-1,1 (Dom),6.83,11.39,50,56.48,1.040,39.73,46.70,44.74,46.40,46.70
6,4-1,1 (Dom),3.90,2.53,70,55.79,0.939,39.68,46.64,44.61,46.40,46.64
7,5-0,1 (Dom),2.79,1.27,70,55.02,0.811,39.61,46.57,44.47,46.40,46.57
8,3-2,1 (Dom),2.30,3.80,70,54.68,1.011,39.58,46.54,44.41,46.40,46.54
9,5-1,1 (Dom),1.96,2.53,70,54.43,0.850,39.56,46.52,44.37,46.40,46.52


Contrôle 1N2 (somme probas scores / outcome) : {'1 (Dom)': np.float64(74.74), '2 (Ext)': np.float64(8.33), 'N (Nul)': np.float64(16.93)}
🏅 Top 3 E[pts 1/2/3] : 2-0 (1.114) | 1-0 (1.113) | 2-1 (1.077)

📊 MATCH 28  mexique - coree du sud — reco : 0-0 (Safe)  |  WR : 46.34%


...

Contrôle 1N2 (somme probas scores / outcome) : {'1 (Dom)': np.float64(46.59), '2 (Ext)': np.float64(23.36), 'N (Nul)': np.float64(30.05)}
🏅 Top 3 E[pts 1/2/3] : 1-0 (0.833) | 2-1 (0.799) | 1-1 (0.745)

📊 MATCH 29  etats unis - australie — reco : 1-0 (Safe)  |  WR : 45.97%


,Score,Outcome,True Proba (%),Crowd cond. (%),Bonus,E[pts MPP],E[pts 1/2/3],WR base (%),WR keep (%),WR x2 (%),WR outcome (%),WR best (%)
0,1-0,1 (Dom),13.15,5.26,50,41.98,1.006,38.94,45.97,42.78,45.38,45.97
1,2-0,1 (Dom),11.69,11.84,50,41.25,0.918,38.88,45.90,42.64,45.38,45.90
2,3-0,1 (Dom),6.89,6.58,50,38.84,0.780,38.66,45.69,42.19,45.38,45.69
3,4-0,1 (Dom),3.12,0.00,100,38.52,0.682,38.64,45.66,42.12,45.38,45.66
4,3-1,1 (Dom),6.15,15.79,50,38.48,0.863,38.63,45.65,42.12,45.38,45.65
5,2-1,1 (Dom),10.15,48.68,20,37.43,0.976,38.53,45.56,41.91,45.38,45.56
6,3-2,1 (Dom),2.78,1.32,70,37.35,0.902,38.53,45.55,41.90,45.38,45.55
7,4-1,1 (Dom),2.75,1.32,70,37.33,0.738,38.53,45.55,41.90,45.38,45.55
8,5-0,1 (Dom),1.11,0.00,100,36.51,0.632,38.46,45.48,41.74,45.38,45.48
9,5-1,1 (Dom),0.96,0.00,100,36.36,0.661,38.44,45.47,41.72,45.38,45.47


Contrôle 1N2 (somme probas scores / outcome) : {'1 (Dom)': np.float64(61.04), '2 (Ext)': np.float64(17.81), 'N (Nul)': np.float64(21.16)}
🏅 Top 3 E[pts 1/2/3] : 1-0 (1.006) | 2-1 (0.976) | 2-0 (0.918)

📊 MATCH 30  ecosse - maroc — reco : 0-1 (Safe)  |  WR : 45.77%


,Score,Outcome,True Proba (%),Crowd cond. (%),Bonus,E[pts MPP],E[pts 1/2/3],WR base (%),WR keep (%),WR x2 (%),WR outcome (%),WR best (%)
0,0-1,2 (Ext),14.57,7.14,50,57.82,0.966,38.73,45.77,44.13,45.11,45.77
1,0-2,2 (Ext),11.09,28.57,30,53.87,0.836,38.37,45.41,43.37,45.11,45.41
2,0-3,2 (Ext),5.82,13.10,50,53.45,0.694,38.33,45.37,43.29,45.11,45.37
3,1-2,2 (Ext),9.47,27.38,30,53.38,0.915,38.33,45.36,43.28,45.11,45.36
4,1-3,2 (Ext),5.01,19.05,50,53.05,0.775,38.30,45.33,43.22,45.11,45.33
5,0-4,2 (Ext),2.41,0.00,100,52.95,0.610,38.29,45.33,43.18,45.11,45.33
6,2-3,2 (Ext),2.25,0.00,100,52.79,0.843,38.28,45.31,43.16,45.11,45.31
7,1-4,2 (Ext),2.01,1.19,70,51.95,0.656,38.20,45.24,43.01,45.11,45.24
8,0-5,2 (Ext),0.78,0.00,100,51.32,0.571,38.14,45.18,42.88,45.11,45.18
9,1-5,2 (Ext),0.62,0.00,100,51.16,0.592,38.13,45.16,42.85,45.11,45.16


Contrôle 1N2 (somme probas scores / outcome) : {'1 (Dom)': np.float64(18.11), '2 (Ext)': np.float64(55.54), 'N (Nul)': np.float64(26.35)}
🏅 Top 3 E[pts 1/2/3] : 0-1 (0.966) | 1-2 (0.915) | 2-3 (0.843)


In [4]:
# ==========================================
# 2. PROJECTION STRATÉGIQUE SUR LES PROCHAINS MATCHS
# ==========================================
print("\n" + "="*50)
print("🔮 PROJECTION STRATÉGIQUE SUR LES PROCHAINS MATCHS")
print("="*50)
print("Analyse de sensibilité : Comment la stratégie évolue-t-elle selon votre dynamique ?\n")

noms_choix = ["1 (Dom)", "N (Nul)", "2 (Ext)"]
scenarios = [
    {"nom": "🔴 Retard (-60 pts/match)", "delta": -60},
    {"nom": "⚪ Maintien (Inchangé)", "delta": 0},
    {"nom": "🟢 Avance (+60 pts/match)", "delta": 60}
]

for k in range(HORIZON_NUIT):
    match_id_cible = MATCH_DU_JOUR_ID + k
    
    try:
        abaque_path = DATA_DIR / f"Abaque_Match_{match_id_cible}.npz"
        data = np.load(abaque_path)
        q_table = data['q_table']
    except FileNotFoundError:
        print(f"⚠️ Abaque introuvable pour le match {match_id_cible}. Fin de la projection.")
        break
        
    print(f"▶️ MATCH {match_id_cible} (Δt = +{k}) :")
    
    for sc in scenarios:
        proj_gap1 = MON_GAP_1 + (sc["delta"] * k)
        proj_gap2 = MON_GAP_2 + (sc["delta"] * k)
        
        g1_idx = max(GAP_MIN, min(GAP_MAX, int(round(proj_gap1)))) + GAP_OFFSET
        g2_idx = max(GAP_MIN, min(GAP_MAX, int(round(proj_gap2)))) + GAP_OFFSET
        
        if HAS_BOOSTER:
            wr_keep = q_table[g1_idx, g2_idx, :, 1]
            wr_use = q_table[g1_idx, g2_idx, :, 2]
            best_keep_idx, best_use_idx = np.argmax(wr_keep), np.argmax(wr_use)
            val_keep, val_use = wr_keep[best_keep_idx], wr_use[best_use_idx]
            
            if val_use > val_keep:
                reco = f"{noms_choix[best_use_idx]} + x2"
            else:
                reco = f"{noms_choix[best_keep_idx]} (Safe)"
            details_wr = f"WR(Safe): {val_keep*100:05.2f}% | WR(x2): {val_use*100:05.2f}%"
        else:
            wr_base = q_table[g1_idx, g2_idx, :, 0]
            best_action = np.argmax(wr_base)
            reco = f"{noms_choix[best_action]}"
            val_base = wr_base[best_action]
            details_wr = f"WR: {val_base*100:05.2f}%"
            
        print(f"  {sc['nom']:<27} | Gaps proj. : {proj_gap1:>4}, {proj_gap2:>4} | Reco : {reco:<14} | {details_wr}")
    print("-" * 90)


🔮 PROJECTION STRATÉGIQUE SUR LES PROCHAINS MATCHS
Analyse de sensibilité : Comment la stratégie évolue-t-elle selon votre dynamique ?

▶️ MATCH 26 (Δt = +0) :
  🔴 Retard (-60 pts/match)    | Gaps proj. : -237,  -71 | Reco : 1 (Dom) (Safe) | WR(Safe): 47.10% | WR(x2): 44.28%
  ⚪ Maintien (Inchangé)       | Gaps proj. : -237,  -71 | Reco : 1 (Dom) (Safe) | WR(Safe): 47.10% | WR(x2): 44.28%
  🟢 Avance (+60 pts/match)    | Gaps proj. : -237,  -71 | Reco : 1 (Dom) (Safe) | WR(Safe): 47.10% | WR(x2): 44.28%
------------------------------------------------------------------------------------------
▶️ MATCH 27 (Δt = +1) :
  🔴 Retard (-60 pts/match)    | Gaps proj. : -297, -131 | Reco : 1 (Dom) (Safe) | WR(Safe): 41.82% | WR(x2): 39.31%
  ⚪ Maintien (Inchangé)       | Gaps proj. : -237,  -71 | Reco : 1 (Dom) (Safe) | WR(Safe): 46.71% | WR(x2): 44.44%
  🟢 Avance (+60 pts/match)    | Gaps proj. : -177,  -11 | Reco : 1 (Dom) (Safe) | WR(Safe): 51.92% | WR(x2): 49.81%
-----------------------------